# 03 · NULL, NaN, safe casts and data-quality quarantine

**Scenario:** Preserve NULLs, distinguish NaN, and capture rejected asset rows with reasons.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Profile missingness and numeric anomalies

NULL means missing; NaN is a floating-point special value. They need different predicates.


In [ ]:
df.agg(*[F.sum(F.col(c).isNull().cast('int')).alias(c+'_nulls') for c in ['distance','fare','pickup_ts']]).show()
checked=df.withColumn('reason',
    F.when(F.col('distance').isNull()|F.isnan('distance'),'missing_distance')
     .when(F.col('distance')<=0,'nonpositive_distance')
     .when(F.col('fare').isNull()|F.isnan('fare'),'missing_fare')
     .when(F.col('fare')<0,'negative_fare'))
good=checked.filter('reason IS NULL').drop('reason')
bad=checked.filter('reason IS NOT NULL')
assert good.count()+bad.count()==df.count()
bad.show(10,truncate=False)


**Expected check:** Valid and rejected counts add to the full asset slice; there may be zero rejections.


## Step 3 — Use safe conversion and explicit defaults

try_cast returns NULL on malformed input under ANSI mode. Do not replace all NULLs when NULL has business meaning.


In [ ]:
converted=df.select('trip_id',F.col('pickup_zip').alias('zip_text'),
    F.expr('try_cast(pickup_zip AS INT)').alias('zip_number'),F.coalesce('fare',F.lit(0.0)).alias('display_fare'))
converted.show(5)
assert converted.count()==df.count()


**Expected check:** Failed ZIP conversions stay NULL; only the display_fare field has a documented zero default.


## Practice

Add a full-record reject for dropoff before pickup. Retain the original fields alongside reason.

## Solution guide

Add a condition comparing dropoff_ts and pickup_ts. Persist the complete bad DataFrame in a course-owned quarantine table, not only a list of IDs.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
